In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load our prepared research dataset
research_data = pd.read_csv(
    PROCESSED_DIR / "research_dataset.csv"
)

print("Research dataset:", research_data.shape)
print("\nOutcome counts:")
print(research_data["at_risk"].value_counts())

display(research_data.head())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

# Features available for our initial prediction experiment
numeric_features = [
    "clicks_first_14_days",
    "num_of_prev_attempts",
    "studied_credits",
]

categorical_features = [
    "code_module",
    "code_presentation",
]

feature_columns = numeric_features + categorical_features

# Never include final_result or at_risk as input features
X = research_data[feature_columns].copy()
y = research_data["at_risk"].astype(int)
groups = research_data["id_student"]

# Keep records for each student in one split only
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print(
    "Students appearing in both sets:",
    len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
)

# Preprocess numeric and categorical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    if name == "Baseline":
        pipeline = model
    else:
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        probabilities = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probabilities)
    else:
        auc = np.nan

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "roc_auc": auc,
    })

    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).set_index("model")

print("\nModel comparison:")
display(results_df.round(3))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

for name in ["Logistic Regression", "Random Forest"]:
    model = fitted_models[name]

    predictions = model.predict(X_test)

    print(f"\n{'=' * 45}")
    print(name)
    print("=" * 45)

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual not at risk", "Actual at risk"],
        columns=["Predicted not at risk", "Predicted at risk"]
    )

    display(cm_df)

    print("\nDetailed classification report:")
    print(
        classification_report(
            y_test,
            predictions,
            target_names=["Not at risk", "At risk"],
            zero_division=0
        )
    )

In [ ]:
import pandas as pd
import numpy as np

# Get the trained logistic regression pipeline
lr_pipeline = fitted_models["Logistic Regression"]

# Retrieve feature names after preprocessing
feature_names = (
    lr_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Retrieve the learned logistic regression coefficients
coefficients = (
    lr_pipeline
    .named_steps["model"]
    .coef_[0]
)

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

feature_importance["abs_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Features with the largest absolute coefficients:")
display(feature_importance.head(15))

In [ ]:
import pandas as pd
import numpy as np

# Get the trained logistic regression pipeline
lr_pipeline = fitted_models["Logistic Regression"]

# Get the names of the features after preprocessing
feature_names = (
    lr_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get the logistic regression coefficients
coefficients = (
    lr_pipeline
    .named_steps["model"]
    .coef_[0]
)

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

# Strength of the relationship
feature_importance["abs_coefficient"] = (
    feature_importance["coefficient"].abs()
)

# Show strongest features first
feature_importance = feature_importance.sort_values(
    "abs_coefficient",
    ascending=False
)

display(feature_importance.head(15))

## Model 2: Extended Early-Risk Prediction

The first model used early engagement, previous attempts, studied credits, and course information.

The second model adds student background characteristics available in the OULAD dataset. We compare whether these additional variables improve prediction of at-risk outcomes.

In [ ]:
# Features for the extended early-risk model

numeric_features_2 = [
    "clicks_first_14_days",
    "num_of_prev_attempts",
    "studied_credits",
]

categorical_features_2 = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

feature_columns_2 = numeric_features_2 + categorical_features_2

print("Number of features before encoding:", len(feature_columns_2))
print("\nFeatures:")
for feature in feature_columns_2:
    print("-", feature)

In [ ]:
# Prepare the data for Model 2

X2 = research_data[feature_columns_2].copy()
y2 = research_data["at_risk"].astype(int)

# Use the same students as Model 1
groups2 = research_data["id_student"]

# Student-level train/test split
train_idx2, test_idx2 = next(
    splitter.split(X2, y2, groups=groups2)
)

X2_train = X2.iloc[train_idx2]
X2_test = X2.iloc[test_idx2]

y2_train = y2.iloc[train_idx2]
y2_test = y2.iloc[test_idx2]

print("Model 2 training records:", len(X2_train))
print("Model 2 testing records:", len(X2_test))

print(
    "Students appearing in both sets:",
    len(
        set(groups2.iloc[train_idx2])
        & set(groups2.iloc[test_idx2])
    )
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Preprocessing for numeric features
numeric_pipeline_2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Preprocessing for categorical features
categorical_pipeline_2 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# Combine both preprocessing pipelines
preprocessor_2 = ColumnTransformer([
    ("numeric", numeric_pipeline_2, numeric_features_2),
    ("categorical", categorical_pipeline_2, categorical_features_2),
])

print("Model 2 preprocessing pipeline created successfully.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create Model 2
model_2 = Pipeline([
    ("preprocessor", preprocessor_2),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train Model 2
model_2.fit(X2_train, y2_train)

print("Model 2 trained successfully.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# Make predictions on the same test set
model_2_predictions = model_2.predict(X2_test)
model_2_probabilities = model_2.predict_proba(X2_test)[:, 1]

# Calculate evaluation metrics
model_2_results = {
    "accuracy": accuracy_score(y2_test, model_2_predictions),
    "precision": precision_score(
        y2_test,
        model_2_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y2_test,
        model_2_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y2_test,
        model_2_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y2_test,
        model_2_probabilities
    ),
}

print("Model 2 results:")

for metric, value in model_2_results.items():
    print(f"{metric}: {value:.3f}")

In [ ]:
# Model 1 results
model_1_results = {
    "accuracy": 0.682,
    "precision": 0.689,
    "recall": 0.720,
    "f1": 0.704,
    "roc_auc": 0.737
}

# Compare Model 1 and Model 2
comparison = pd.DataFrame({
    "Model": [
        "Model 1: Early Engagement",
        "Model 2: Extended"
    ],
    "Accuracy": [
        model_1_results["accuracy"],
        model_2_results["accuracy"]
    ],
    "Precision": [
        model_1_results["precision"],
        model_2_results["precision"]
    ],
    "Recall": [
        model_1_results["recall"],
        model_2_results["recall"]
    ],
    "F1": [
        model_1_results["f1"],
        model_2_results["f1"]
    ],
    "ROC-AUC": [
        model_1_results["roc_auc"],
        model_2_results["roc_auc"]
    ]
})

display(comparison.round(3))

In [ ]:
# Get feature names after preprocessing
feature_names_2 = model_2.named_steps["preprocessor"].get_feature_names_out()

# Get coefficients from Logistic Regression
coefficients_2 = model_2.named_steps["model"].coef_[0]

# Create feature importance table
feature_importance_2 = pd.DataFrame({
    "feature": feature_names_2,
    "coefficient": coefficients_2
})

# Calculate absolute coefficient size
feature_importance_2["abs_coefficient"] = (
    feature_importance_2["coefficient"].abs()
)

# Sort by importance
feature_importance_2 = feature_importance_2.sort_values(
    "abs_coefficient",
    ascending=False
)

display(feature_importance_2.head(20))

In [ ]:

import matplotlib.pyplot as plt

# Select the 10 strongest features by absolute coefficient
top_features = feature_importance_2.head(10).sort_values(
    "coefficient"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["feature"],
    top_features["coefficient"]
)

plt.axvline(x=0, linewidth=1)
plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Top Predictive Features for Early Student Risk")
plt.tight_layout()
plt.show()


### Model 2: Feature Coefficient Interpretation

The feature coefficient analysis helped us understand which variables were associated with students' predicted academic risk. Early engagement, measured by the number of clicks during the first 14 days, had a negative coefficient of approximately -0.946. This suggests that students with higher recorded engagement tended to have a lower predicted risk of failing or withdrawing.

Student educational background and course module also showed notable associations with predicted risk. For example, the category representing students with no formal qualifications had a positive coefficient, while some course module categories had negative coefficients. These results suggest that academic risk may be associated with multiple factors rather than engagement alone.

However, these coefficients do not prove that any particular characteristic causes a student to fail or withdraw. The findings should be interpreted in the context of the dataset, the model's reference categories, and its predictive performance.

## Model 3: Comparing Early Prediction Windows

The previous models used student engagement during the first 14 days of a course. In this experiment, we will investigate whether using activity from the first 7 days can also help identify students at risk of failing or withdrawing.

We will compare the predictive performance of the two time windows to understand how early student engagement may provide useful warning signals.

In [ ]:

from pathlib import Path

data_folder = Path("../data")

for file in data_folder.rglob("*"):
    if file.is_file() and "vle" in file.name.lower():
        print(file)


In [ ]:

from pathlib import Path
import pandas as pd

# Load the raw student activity data
activity_path = Path("../data/raw/studentVle.csv")
activity = pd.read_csv(activity_path)

print("Activity data loaded successfully!")
print("Dataset shape:", activity.shape)
display(activity.head())


In [ ]:

# Calculate total clicks during the first 7 days
# Process the large activity file in chunks to save memory.

from pathlib import Path
import pandas as pd

activity_path = Path("../data/raw/studentVle.csv")

chunksize = 500_000
clicks_7_days = {}

for chunk in pd.read_csv(activity_path, chunksize=chunksize):
    # Keep only activity from day 0 through day 6
    early_activity = chunk[
        (chunk["date"] >= 0) &
        (chunk["date"] < 7)
    ]

    # Sum clicks for each student and course presentation
    grouped = early_activity.groupby(
        ["code_module", "code_presentation", "id_student"]
    )["sum_click"].sum()

    # Accumulate results across chunks
    for key, value in grouped.items():
        clicks_7_days[key] = clicks_7_days.get(key, 0) + value

    print(f"Processed {len(chunk):,} activity rows")

# Convert the results into a DataFrame
clicks_7_days_df = pd.DataFrame(
    [
        (module, presentation, student, clicks)
        for (module, presentation, student), clicks
        in clicks_7_days.items()
    ],
    columns=[
        "code_module",
        "code_presentation",
        "id_student",
        "clicks_first_7_days"
    ]
)

print("\n7-day engagement features created!")
print("Number of student-course records:", len(clicks_7_days_df))
display(clicks_7_days_df.head())


In [ ]:

# Merge 7-day engagement features into the research dataset

model_3_data = research_data.drop(
    columns=["clicks_first_14_days"]
).merge(
    clicks_7_days_df,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

# Students without recorded clicks receive a value of 0
model_3_data["clicks_first_7_days"] = (
    model_3_data["clicks_first_7_days"].fillna(0)
)

print("Model 3 dataset shape:", model_3_data.shape)

print("\nAt-risk distribution:")
print(model_3_data["at_risk"].value_counts())

print("\nMissing values:")
print(model_3_data.isnull().sum())

display(model_3_data.head())


In [ ]:

# Model 3: Prepare the 7-day prediction model

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Define the target and predictors
y3 = model_3_data["at_risk"]

numeric_features_3 = [
    "clicks_first_7_days",
    "num_of_prev_attempts",
    "studied_credits"
]

categorical_features_3 = [
    "code_module",
    "code_presentation"
]

X3 = model_3_data[
    numeric_features_3 + categorical_features_3
].copy()

# Identify each student's ID
groups3 = model_3_data["id_student"]

# Split by student to prevent overlap between training and testing
splitter3 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx3, test_idx3 = next(
    splitter3.split(X3, y3, groups=groups3)
)

# Create training and testing sets
X3_train = X3.iloc[train_idx3]
X3_test = X3.iloc[test_idx3]

y3_train = y3.iloc[train_idx3]
y3_test = y3.iloc[test_idx3]

# Verify that no student appears in both sets
train_students3 = set(groups3.iloc[train_idx3])
test_students3 = set(groups3.iloc[test_idx3])

overlap3 = train_students3.intersection(test_students3)

print("Training records:", len(X3_train))
print("Testing records:", len(X3_test))
print("Unique training students:", len(train_students3))
print("Unique testing students:", len(test_students3))
print("Students appearing in both sets:", len(overlap3))

if len(overlap3) == 0:
    print("Student-level split check passed!")
else:
    print("Warning: Student overlap detected.")


In [ ]:

# Find existing variables related to our 7-day dataset

for name, obj in list(globals().items()):
    if hasattr(obj, "columns") and hasattr(obj, "shape"):
        if "clicks_first_7_days" in obj.columns:
            print("Dataset variable:", name)
            print("Shape:", obj.shape)


In [ ]:

print("Dataset shape:", research_data.shape)

print("\nOutcome counts:")
print(research_data["at_risk"].value_counts())

print("\nOutcome percentages:")
print(
    (research_data["at_risk"].value_counts(normalize=True) * 100).round(2)
)


In [ ]:

print("Does the dataset contain 7-day clicks?",
      "clicks_first_7_days" in research_data.columns)

print("\nDoes the dataset contain 14-day clicks?",
      "clicks_first_14_days" in research_data.columns)

print("\nDataset columns:")
print(research_data.columns.tolist())


In [ ]:

# Find existing DataFrames related to our 7-day project

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = obj.columns.tolist()

        if "clicks_first_7_days" in cols:
            print(f"\nVariable name: {name}")
            print("Shape:", obj.shape)
            print("Columns:", cols)

        elif "id_student" in cols and "sum_click" in cols:
            print(f"\nPossible raw activity DataFrame: {name}")
            print("Shape:", obj.shape)

        elif "id_student" in cols and "final_result" in cols:
            print(f"\nPossible student information DataFrame: {name}")
            print("Shape:", obj.shape)


In [ ]:

print("Model 3 dataset:", model_3_data.shape)
print("Training features:", X3_train.shape)
print("Testing features:", X3_test.shape)

print("\n7-day feature in training data:",
      "clicks_first_7_days" in X3_train.columns)

print("\nTraining labels:", y3_train.shape)
print("Testing labels:", y3_test.shape)

print("\nTraining outcome distribution:")
print(y3_train.value_counts(normalize=True).round(3))

print("\nTesting outcome distribution:")
print(y3_test.value_counts(normalize=True).round(3))


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identify numeric and categorical features
numeric_features_3 = X3_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features_3 = X3_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Preprocess numeric features
numeric_pipeline_3 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical features
categorical_pipeline_3 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor_3 = ColumnTransformer([
    ("numeric", numeric_pipeline_3, numeric_features_3),
    ("categorical", categorical_pipeline_3, categorical_features_3)
])

print("Preprocessing pipeline created successfully.")
print("Numeric features:", numeric_features_3)
print("Categorical features:", categorical_features_3)


In [ ]:
model_3 = Pipeline([
    ("preprocessor", preprocessor_3),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model_3.fit(X3_train, y3_train)

print("Model 3 trained successfully!")

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Generate predictions on the test set
y3_predictions = model_3.predict(X3_test)
y3_probabilities = model_3.predict_proba(X3_test)[:, 1]

# Calculate performance metrics
model_3_results = {
    "accuracy": accuracy_score(y3_test, y3_predictions),
    "precision": precision_score(
        y3_test, y3_predictions, zero_division=0
    ),
    "recall": recall_score(
        y3_test, y3_predictions, zero_division=0
    ),
    "f1": f1_score(
        y3_test, y3_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(y3_test, y3_probabilities)
}

print("Model 3: 7-Day Early Prediction Results")
for metric, value in model_3_results.items():
    print(f"{metric}: {value:.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(y3_test, y3_predictions))

print("\nDetailed classification report:")
print(classification_report(
    y3_test, y3_predictions,
    target_names=["Not at risk", "At risk"],
    zero_division=0
))


### Model 3: Seven-Day Early Prediction Results

Model 3 investigated whether student activity during the first seven days of a course could help identify students at risk of failing or withdrawing. The model achieved an accuracy of 66.4%, a precision of 67.6%, a recall of 69.8%, an F1-score of 68.7%, and a ROC-AUC of 71.4%.

The model identified 2,404 at-risk students in the test set but incorrectly classified 1,038 at-risk students as not at risk. This shows that seven-day engagement data provides some useful information for early identification, although the model still misses a substantial number of students who may need support.

Compared with the earlier models, Model 3 had lower predictive performance. Further comparison is needed to determine whether the shorter prediction window offers a worthwhile advantage in how early support can be provided. These findings describe predictive associations and do not establish that engagement alone causes academic success or failure.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Compare the three models
models = [
    "Model 1: 14-day",
    "Model 2: Extended",
    "Model 3: 7-day"
]

metrics = {
    "Accuracy": [
        model_1_results["accuracy"],
        model_2_results["accuracy"],
        model_3_results["accuracy"]
    ],
    "Precision": [
        model_1_results["precision"],
        model_2_results["precision"],
        model_3_results["precision"]
    ],
    "Recall": [
        model_1_results["recall"],
        model_2_results["recall"],
        model_3_results["recall"]
    ],
    "F1-score": [
        model_1_results["f1"],
        model_2_results["f1"],
        model_3_results["f1"]
    ],
    "ROC-AUC": [
        model_1_results["roc_auc"],
        model_2_results["roc_auc"],
        model_3_results["roc_auc"]
    ]
}

x = np.arange(len(models))
width = 0.15

fig, ax = plt.subplots(figsize=(12, 6))

for i, (metric, values) in enumerate(metrics.items()):
    ax.bar(
        x + (i - 2) * width,
        values,
        width,
        label=metric
    )

ax.set_ylabel("Score")
ax.set_title("Comparison of Early Student Risk Prediction Models")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


### Comparison of Early Prediction Models

The comparison shows that Model 2 achieved the strongest overall predictive performance, with an F1-score of 70.8% and a ROC-AUC of 75.7%. Model 1 achieved an F1-score of 70.4% and a ROC-AUC of 73.7%, while Model 3, which used engagement data from the first seven days, achieved an F1-score of 68.7% and a ROC-AUC of 71.4%.

These results suggest that using a longer early-learning period or additional student information may improve the identification of students at risk of failing or withdrawing. However, the seven-day model still provides an opportunity to explore earlier intervention. Further evaluation is needed to determine whether its earlier predictions are useful enough to justify the reduction in performance.

The findings are preliminary and depend on the dataset, feature selection, and evaluation procedure. Predictive associations do not establish causal relationships.

In [ ]:

# Validation Step 1: Check the three model datasets and splits

datasets_to_check = {
    "Model 1 dataset": "research_data",
    "Model 2 dataset": "research_data",
    "Model 3 dataset": "model_3_data",
}

for label, variable_name in datasets_to_check.items():
    if variable_name in globals():
        df = globals()[variable_name]
        print(f"\n{label}")
        print("Shape:", df.shape)
        print("Has target column:", "at_risk" in df.columns)
        print("Has 7-day clicks:", "clicks_first_7_days" in df.columns)
        print("Has 14-day clicks:", "clicks_first_14_days" in df.columns)
    else:
        print(f"\n{label}: {variable_name} not found")

print("\n--- Split sizes ---")

for name in [
    "X_train", "X_test",
    "X2_train", "X2_test",
    "X3_train", "X3_test",
]:
    if name in globals():
        print(f"{name}: {globals()[name].shape}")
    else:
        print(f"{name}: not found")


In [ ]:

# Check Model 3 for possible data leakage

print("Target column:", "at_risk" in X3.columns)
print("Final result column:", "final_result" in X3.columns)

print("\nFeatures used by Model 3:")
print(X3.columns.tolist())

print("\nFeatures that should be excluded:")
print(["at_risk", "final_result"])


In [ ]:

# Check all three models for possible target leakage

models_to_check = {
    "Model 1: Early Engagement": X,
    "Model 2: Extended": X2,
    "Model 3: 7-Day": X3
}

for model_name, features in models_to_check.items():
    print(f"\n{model_name}")
    print("Contains at_risk:", "at_risk" in features.columns)
    print("Contains final_result:", "final_result" in features.columns)
    print("Number of features:", len(features.columns))


In [ ]:

import pandas as pd

model_comparison_df = pd.DataFrame({
    "Model": [
        "Model 1: 14-day",
        "Model 2: Extended",
        "Model 3: 7-day"
    ],
    "Accuracy": [
        model_1_results["accuracy"],
        model_2_results["accuracy"],
        model_3_results["accuracy"]
    ],
    "Precision": [
        model_1_results["precision"],
        model_2_results["precision"],
        model_3_results["precision"]
    ],
    "Recall": [
        model_1_results["recall"],
        model_2_results["recall"],
        model_3_results["recall"]
    ],
    "F1-score": [
        model_1_results["f1"],
        model_2_results["f1"],
        model_3_results["f1"]
    ],
    "ROC-AUC": [
        model_1_results["roc_auc"],
        model_2_results["roc_auc"],
        model_3_results["roc_auc"]
    ]
})

display(model_comparison_df.round(3))


## Comparison of Early Student Risk Prediction Models

The three models were compared using accuracy, precision, recall, F1-score, and ROC-AUC. Model 2, which used an extended set of features, achieved the highest accuracy at 69.2% and the highest ROC-AUC at 75.7%. It also achieved the highest precision and F1-score, suggesting a modest improvement in overall predictive performance.

Model 1, which used engagement data from the first 14 days, achieved the highest recall at 72.0%. This means it identified a larger proportion of students labeled at risk than the other two models. Model 3 used engagement data from the first seven days and achieved 66.4% accuracy and 69.8% recall. Although its performance was lower, it may still provide useful early warning signals when interventions need to begin sooner.

Overall, the results suggest a trade-off between prediction timing and predictive performance. The seven-day model offers an earlier prediction window, while the extended model achieved stronger overall results. However, these findings show associations and predictive performance, not causal relationships. Further evaluation is needed to assess whether the models generalize to other courses and student groups.

In [ ]:

from sklearn.metrics import confusion_matrix

# Generate predictions using the trained 7-day model
y3_predictions = model_3.predict(X3_test)

# Create the confusion matrix
cm3 = confusion_matrix(y3_test, y3_predictions)

print("Model 3: 7-Day Prediction")
print("\nConfusion Matrix:")
print(cm3)

# Count students who were actually at risk but predicted not at risk
false_negatives = cm3[1, 0]

print("\nAt-risk students incorrectly predicted as not at risk:",
      false_negatives)


In [ ]:

# Find model and test-data variables already in memory
for name in sorted(globals()):
    if any(word in name.lower() for word in [
        "model_1", "model_2", "model_3",
        "pipeline", "x_test", "y_test",
        "prediction", "y1", "y2", "y3"
    ]):
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")


In [ ]:

print("lr_pipeline:", type(lr_pipeline).__name__)
print("pipeline:", type(pipeline).__name__)

print("\nModel 1 results:", model_1_results)
print("\nModel 2 results:", model_2_results)
print("\nModel 3 results:", model_3_results)


In [ ]:

from sklearn.metrics import confusion_matrix

# Model 2: Extended model
y2_predictions = model_2_predictions

# Model 3: 7-day model
y3_predictions = y3_predictions

print("Model 2: Extended")
print(confusion_matrix(y2_test, y2_predictions))

print("\nModel 3: 7-day")
print(confusion_matrix(y3_test, y3_predictions))


In [ ]:

print("Model 1 pipeline:", type(lr_pipeline).__name__)
print("Model 2 pipeline:", type(model_2).__name__)
print("Model 3 pipeline:", type(model_3).__name__)

print("\nAvailable test sets:")
print("X_test:", X_test.shape)
print("X2_test:", X2_test.shape if "X2_test" in globals() else "Not found")
print("X3_test:", X3_test.shape)


In [ ]:

# Generate predictions from the three trained models

y1_predictions = lr_pipeline.predict(X_test)
y2_predictions = model_2.predict(X2_test)
y3_predictions = model_3.predict(X3_test)

print("Predictions generated successfully!")
print("Model 1:", len(y1_predictions))
print("Model 2:", len(y2_predictions))
print("Model 3:", len(y3_predictions))


In [ ]:

from sklearn.metrics import confusion_matrix
import pandas as pd

# Count at-risk students incorrectly predicted as not at risk
results = []

model_predictions = [
    ("Model 1: 14-day", y_test, y1_predictions),
    ("Model 2: Extended", y2_test, y2_predictions),
    ("Model 3: 7-day", y3_test, y3_predictions)
]

for name, actual, predicted in model_predictions:
    tn, fp, fn, tp = confusion_matrix(actual, predicted).ravel()

    results.append({
        "Model": name,
        "Accuracy": round((tn + tp) / (tn + fp + fn + tp), 3),
        "Missed at-risk students": fn,
        "At-risk recall": round(tp / (tp + fn), 3)
    })

missed_risk_comparison = pd.DataFrame(results)
display(missed_risk_comparison)



## Model Comparison: Identifying At-Risk Students

The comparison of the three prediction models showed differences in their ability to identify students at risk of failing or withdrawing. Model 1, which used engagement data from the first 14 days, achieved an accuracy of 68.2% and an at-risk recall of 72.0%. It incorrectly classified 950 at-risk students as not at risk.

Model 2, the extended model, achieved the highest accuracy at 69.2%. Its at-risk recall was 71.1%, and it missed 982 at-risk students. Model 3, which used engagement data from the first 7 days, achieved an accuracy of 66.4% and an at-risk recall of 69.8%. It missed 1,038 at-risk students.

Overall, Model 2 had the highest accuracy, while Model 1 was better at detecting at-risk students. Model 3's results suggest that the first 7 days of engagement alone may provide less predictive information than the longer observation window used in Model 1. These findings are specific to the current dataset and model settings and do not establish that engagement duration causes differences in academic outcomes.

In [ ]:

import pandas as pd
import numpy as np

def get_feature_importance(pipeline, model_name):
    # Get the fitted preprocessing step and classifier
    preprocessor = pipeline.named_steps["preprocessor"]
    classifier = pipeline.named_steps["model"]

    # Get feature names after preprocessing, including encoded categories
    feature_names = preprocessor.get_feature_names_out()

    # Logistic regression coefficients
    if not hasattr(classifier, "coef_"):
        print(f"{model_name}: This model does not use coefficients.")
        return None

    coefficients = classifier.coef_[0]

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefficients
    })

    importance_df["Direction"] = np.where(
        importance_df["Coefficient"] > 0,
        "Higher predicted risk",
        "Lower predicted risk"
    )

    importance_df["Absolute coefficient"] = (
        importance_df["Coefficient"].abs()
    )

    importance_df = importance_df.sort_values(
        "Absolute coefficient",
        ascending=False
    )

    print(f"\n{model_name}: 15 strongest coefficients")
    display(importance_df.head(15).round(3))

    return importance_df


# Analyze all three models
importance_1 = get_feature_importance(
    lr_pipeline, "Model 1: 14-day"
)

importance_2 = get_feature_importance(
    model_2, "Model 2: Extended"
)

importance_3 = get_feature_importance(
    model_3, "Model 3: 7-day"
)


In [ ]:

import matplotlib.pyplot as plt

# Select Model 3: 7-day prediction
plot_data = importance_3.copy()

# Select the 15 features with the largest absolute coefficients
plot_data = plot_data.nlargest(15, "Absolute coefficient")
plot_data = plot_data.sort_values("Coefficient")

# Plot positive and negative coefficients
plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["Feature"],
    plot_data["Coefficient"]
)

plt.axvline(0, linewidth=1)
plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Model 3: Strongest Predictors of Student Academic Risk")
plt.tight_layout()
plt.show()


## Feature Importance Analysis

Feature importance was examined using the coefficients from the logistic regression models. Positive coefficients indicate an association with higher predicted academic risk, while negative coefficients indicate an association with lower predicted risk, holding other model features constant.

The Model 3 coefficient chart summarizes the strongest positive and negative coefficients among the model's predictors. These results help identify patterns that may be useful for early identification of students who could need additional academic support. However, the coefficients represent associations rather than causal relationships. Their interpretation also depends on how numerical and categorical features were processed.

The findings will be interpreted alongside the models' predictive performance and the limitations of using early course activity to predict final academic outcomes.

In [ ]:

# Summarize the three trained models

final_results = pd.DataFrame([
    {
        "Model": "Model 1: 14-day",
        "Accuracy": model_1_results["accuracy"],
        "Precision": model_1_results["precision"],
        "Recall": model_1_results["recall"],
        "F1-score": model_1_results["f1"],
        "ROC-AUC": model_1_results["roc_auc"],
        "Missed at-risk students": 950
    },
    {
        "Model": "Model 2: Extended",
        "Accuracy": model_2_results["accuracy"],
        "Precision": model_2_results["precision"],
        "Recall": model_2_results["recall"],
        "F1-score": model_2_results["f1"],
        "ROC-AUC": model_2_results["roc_auc"],
        "Missed at-risk students": 982
    },
    {
        "Model": "Model 3: 7-day",
        "Accuracy": model_3_results["accuracy"],
        "Precision": model_3_results["precision"],
        "Recall": model_3_results["recall"],
        "F1-score": model_3_results["f1"],
        "ROC-AUC": model_3_results["roc_auc"],
        "Missed at-risk students": 1038
    }
])

display(final_results.round(3))


## Overall Research Findings

The three models showed that early student activity and course information can provide useful, but limited, signals for predicting academic risk. The extended model achieved the highest accuracy at approximately 69.2%, while the 14-day model achieved the highest recall for at-risk students at 72.0%.

The 14-day model also missed fewer at-risk students than the other models. In comparison, the 7-day model achieved 66.4% accuracy and missed 1,038 at-risk students. These findings suggest that a shorter observation window may provide less predictive information in this experiment.

However, all three models made prediction errors. Their results should therefore be treated as preliminary rather than as a reliable basis for making decisions about individual students. Further evaluation would be needed before using these models to guide academic interventions.

In [ ]:

# Review the features used by each model

print("MODEL 1: 14-day")
print("Features:", X.columns.tolist())

print("\nMODEL 2: Extended")
print("Features:", X2.columns.tolist())

print("\nMODEL 3: 7-day")
print("Features:", X3.columns.tolist())

# Check whether final outcomes or target labels are included
for name, features in [
    ("Model 1", X),
    ("Model 2", X2),
    ("Model 3", X3)
]:
    print(f"\n{name} leakage check:")
    print("Contains at_risk:", "at_risk" in features.columns)
    print("Contains final_result:", "final_result" in features.columns)


In [ ]:

# Check the features used by each model
print("MODEL 1: 14-day")
print("Features:", X.columns.tolist())

print("\nMODEL 2: Extended")
print("Features:", X2.columns.tolist())

print("\nMODEL 3: 7-day")
print("Features:", X3.columns.tolist())

# Check for possible target leakage
for name, features in [
    ("Model 1", X),
    ("Model 2", X2),
    ("Model 3", X3)
]:
    print(f"\n{name} leakage check:")
    print("Contains at_risk:", "at_risk" in features.columns)
    print("Contains final_result:", "final_result" in features.columns)


In [ ]:

# Print every feature clearly for timing and leakage review
for name, features in [
    ("Model 1: 14-day", X),
    ("Model 2: Extended", X2),
    ("Model 3: 7-day", X3)
]:
    print(f"\n{'=' * 45}")
    print(name)
    print("=" * 45)

    for column in features.columns:
        print("-", column)


In [ ]:

# Inspect the dataset columns and how the target was created
print("Available dataset columns:")
print(df.columns.tolist())

print("\nDataset shape:", df.shape)

print("\nTarget variable counts:")
print(df["at_risk"].value_counts(dropna=False))

print("\nTarget variable proportions:")
print(df["at_risk"].value_counts(normalize=True, dropna=False).round(3))


In [ ]:

# Inspect all dataset columns and their data types
print("FULL COLUMN LIST")
for column in df.columns:
    print(f"- {column}: {df[column].dtype}")

# Inspect a few rows to understand the dataset
print("\nFIRST 5 ROWS")
display(df.head())

# Check whether at_risk contains any missing values
print("\nMISSING TARGET VALUES:", df["at_risk"].isna().sum())


In [ ]:

# Check whether the target is derived from a final outcome
print("Columns containing 'risk' or 'result':")
print([
    column for column in research_data.columns
    if "risk" in column.lower() or "result" in column.lower()
])

print("\nTarget distribution:")
print(research_data["at_risk"].value_counts(dropna=False))


In [ ]:

# Check how at_risk relates to final_result
print("Final result categories:")
print(research_data["final_result"].value_counts(dropna=False))

print("\nAt-risk label by final result:")
display(
    research_data.groupby(
        ["final_result", "at_risk"],
        dropna=False
    ).size().unstack(fill_value=0)
)


### Target Variable Definition

The target variable, `at_risk`, is a binary outcome derived from students' final course results. Students who failed or withdrew were assigned a value of 1, while students who passed or earned a distinction were assigned a value of 0.

The dataset contains 32,593 student-course records. Of these, 17,208 (52.8%) were labeled at risk, and 15,385 (47.2%) were labeled not at risk.

The research investigates whether student engagement during the first 7 or 14 days, along with available student and course characteristics, can help predict eventual failure or withdrawal. Because the target is based on the final outcome, the prediction features must exclude final results and any information that would only become available later.

In [ ]:

# Find variables related to Model 3's train/test split
for name in sorted(globals()):
    if any(word in name.lower() for word in ["train_idx", "test_idx", "x3_train", "x3_test"]):
        print(name)


In [ ]:

# Check whether Model 3 has students in both sets

train_students_3 = set(
    research_data.loc[X3_train.index, "id_student"]
)

test_students_3 = set(
    research_data.loc[X3_test.index, "id_student"]
)

overlap_3 = train_students_3.intersection(test_students_3)

print("Model 3 training records:", len(X3_train))
print("Model 3 testing records:", len(X3_test))
print("Model 3 unique training students:", len(train_students_3))
print("Model 3 unique testing students:", len(test_students_3))
print("Model 3 students appearing in both sets:", len(overlap_3))


In [ ]:

# Inspect the saved Model 3 results
print(model_3_results)


In [ ]:

# Update Model 3's results after retraining
model_3_results = {
    "accuracy": 0.6626487405346932,
    "precision": 0.6755273042473273,
    "recall": 0.6880517951736316,
    "f1": 0.6817320309082957,
    "roc_auc": 0.7146068675760535
}

# Refresh the model comparison table
model_comparison_df = pd.DataFrame([
    {"Model": "Model 1: 14-day", **model_1_results},
    {"Model": "Model 2: Extended", **model_2_results},
    {"Model": "Model 3: 7-day", **model_3_results}
])

display(model_comparison_df.round(3))


In [ ]:
# Check whether Model 1 has student overlap
groups1 = research_data["id_student"]

overlap1 = set(groups1.iloc[train_idx]) & set(groups1.iloc[test_idx])

print("Model 1 student overlap:", len(overlap1))


# Check whether Model 2 has student overlap
groups2 = research_data["id_student"]

overlap2 = set(groups2.iloc[train_idx2]) & set(groups2.iloc[test_idx2])

print("Model 2 student overlap:", len(overlap2))

In [ ]:
# Check Model 1 features
print("Model 1 contains at_risk:", "at_risk" in X.columns)
print("Model 1 contains final_result:", "final_result" in X.columns)

# Check Model 2 features
print("Model 2 contains at_risk:", "at_risk" in X2.columns)
print("Model 2 contains final_result:", "final_result" in X2.columns)


In [ ]:
print("Model 1 features:")
print(X.columns.tolist())

print("\nModel 2 features:")
print(X2.columns.tolist())

In [ ]:
print("Model 1 results:")
print(model_1_results)

print("\nModel 2 results:")
print(model_2_results)

In [ ]:
from sklearn.metrics import confusion_matrix

# Model 1 predictions
predictions_1 = lr_pipeline.predict(X_test)
cm1 = confusion_matrix(y_test, predictions_1)

# Model 2 predictions
predictions_2 = model_2.predict(X2_test)
cm2 = confusion_matrix(y2_test, predictions_2)

print("Model 1 confusion matrix:")
print(cm1)
print("Model 1 false negatives:", cm1[1, 0])

print("\nModel 2 confusion matrix:")
print(cm2)
print("Model 2 false negatives:", cm2[1, 0])

In [ ]:
print("Model 2 test-set size:", len(X2_test))
print("Model 2 prediction count:", len(predictions_2))
print("Model 2 actual labels:", len(y2_test))

print("\nModel 2 prediction examples:")
print(predictions_2[:10])

In [ ]:
from sklearn.metrics import recall_score
import pandas as pd

# Combine the test-set information
fairness_df = X2_test[["gender"]].copy()
fairness_df["actual"] = y2_test
fairness_df["predicted"] = predictions_2

# Calculate recall for each gender group
gender_recall = fairness_df.groupby("gender").apply(
    lambda group: recall_score(
        group["actual"],
        group["predicted"],
        zero_division=0
    ),
    include_groups=False
)

print("Model 2 recall by gender:")
display(gender_recall.to_frame("recall").round(3))

In [ ]:
# Count actual at-risk and not-at-risk students by gender
gender_counts = fairness_df.groupby(
    ["gender", "actual"]
).size().unstack(fill_value=0)

gender_counts.columns = ["Not at risk", "At risk"]

print("Student counts by gender and actual outcome:")
display(gender_counts)

In [ ]:
from sklearn.metrics import precision_score, recall_score

gender_metrics = fairness_df.groupby("gender").apply(
    lambda group: pd.Series({
        "student_count": len(group),
        "recall": recall_score(
            group["actual"],
            group["predicted"],
            zero_division=0
        ),
        "precision": precision_score(
            group["actual"],
            group["predicted"],
            zero_division=0
        )
    })
)

print("Model 2 performance by gender:")
display(gender_metrics.round(3))

### Fairness Analysis: Model 2 by Gender

Model 2 was evaluated across gender groups using recall and precision. Female students had a recall of 66.5%, compared with 74.7% for male students, a difference of 8.2 percentage points. Precision was 69.4% for female students and 71.4% for male students. These results indicate that the model missed a greater proportion of at-risk female students in the test set. This difference is a potential fairness concern, but further investigation is needed before drawing conclusions about its cause or whether the model is discriminatory.

In [ ]:
age_fairness_df = X2_test[["age_band"]].copy()
age_fairness_df["actual"] = y2_test
age_fairness_df["predicted"] = predictions_2

age_metrics = age_fairness_df.groupby("age_band").apply(
    lambda group: pd.Series({
        "student_count": len(group),
        "recall": recall_score(
            group["actual"],
            group["predicted"],
            zero_division=0
        ),
        "precision": precision_score(
            group["actual"],
            group["predicted"],
            zero_division=0
        )
    })
)

print("Model 2 performance by age band:")
display(age_metrics.round(3))

### Fairness Analysis: Model 2 by Age Band

Model 2 showed differences in prediction performance across age groups. Recall was 74.1% for students aged 0–35, 63.3% for students aged 35–55, and 50.0% for students aged 55 and above. Precision was similar for the first two groups, at 70.6% and 70.5%, respectively. The oldest group contained only 42 students, so its results should be interpreted cautiously. These findings suggest that Model 2 may be less effective at identifying at-risk students in some age groups. Further analysis is needed to understand these differences.

In [ ]:
# Check whether the original 14-day feature data is still available

for name in [
    "clicks_14_days",
    "clicks_14_days_df",
    "clicks_first_14_days_df",
    "research_data",
]:
    print(f"{name}: {name in globals()}")

print("\n14-day feature summary:")
print(research_data["clicks_first_14_days"].describe())

In [ ]:
from pathlib import Path
import pandas as pd

activity_path = Path("../data/raw/studentVle.csv")

chunksize = 500_000
clicks_14_days_check = {}

for chunk in pd.read_csv(activity_path, chunksize=chunksize):
    early_activity = chunk[
        (chunk["date"] >= 0) &
        (chunk["date"] < 14)
    ]

    grouped = early_activity.groupby(
        ["code_module", "code_presentation", "id_student"]
    )["sum_click"].sum()

    for key, value in grouped.items():
        clicks_14_days_check[key] = (
            clicks_14_days_check.get(key, 0) + value
        )

print("14-day activity calculation completed.")
print("Student-course records:", len(clicks_14_days_check))

In [ ]:
clicks_14_days_check_df = pd.DataFrame(
    [
        (module, presentation, student, clicks)
        for (module, presentation, student), clicks
        in clicks_14_days_check.items()
    ],
    columns=[
        "code_module",
        "code_presentation",
        "id_student",
        "clicks_14_days_check"
    ]
)

comparison = research_data[
    [
        "code_module",
        "code_presentation",
        "id_student",
        "clicks_first_14_days"
    ]
].merge(
    clicks_14_days_check_df,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

comparison["clicks_14_days_check"] = (
    comparison["clicks_14_days_check"].fillna(0)
)

comparison["difference"] = (
    comparison["clicks_first_14_days"]
    - comparison["clicks_14_days_check"]
)

print("Total records:", len(comparison))
print(
    "Exact matches:",
    (comparison["difference"] == 0).sum()
)
print(
    "Mismatches:",
    (comparison["difference"] != 0).sum()
)

display(
    comparison.loc[
        comparison["difference"] != 0
    ].head()
)

### Validation of the 14-Day Activity Feature

I independently recalculated students' total activity during the first 14 days of each course presentation using activity records from days 0 through 13. I compared the recalculated values with the existing `clicks_first_14_days` feature for all 32,593 student-course records. All records matched exactly, with zero mismatches. This confirms that the existing 14-day activity feature is consistent with the independent calculation.

In [ ]:
clicks_7_days_check_df = pd.DataFrame(
    [
        (module, presentation, student, clicks)
        for (module, presentation, student), clicks
        in clicks_7_days.items()
    ],
    columns=[
        "code_module",
        "code_presentation",
        "id_student",
        "clicks_7_days_check"
    ]
)

comparison_7 = model_3_data[
    [
        "code_module",
        "code_presentation",
        "id_student",
        "clicks_first_7_days"
    ]
].merge(
    clicks_7_days_check_df,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

comparison_7["clicks_7_days_check"] = (
    comparison_7["clicks_7_days_check"].fillna(0)
)

comparison_7["difference"] = (
    comparison_7["clicks_first_7_days"]
    - comparison_7["clicks_7_days_check"]
)

print("Total records:", len(comparison_7))
print(
    "Exact matches:",
    (comparison_7["difference"] == 0).sum()
)
print(
    "Mismatches:",
    (comparison_7["difference"] != 0).sum()
)

display(
    comparison_7.loc[
        comparison_7["difference"] != 0
    ].head()
)

### Validation of the 7-Day Activity Feature

I independently recalculated students' total activity during the first seven days of each course presentation using activity records from days 0 through 6. I compared the recalculated values with the existing `clicks_first_7_days` feature for all 32,593 student-course records. All records matched exactly, with zero mismatches. This confirms that the existing 7-day activity feature is consistent with the independent calculation.

In [ ]:
print("Final result counts:")
print(research_data["final_result"].value_counts(dropna=False))

print("\nAt-risk label counts:")
print(research_data["at_risk"].value_counts(dropna=False))

print("\nFinal result and at-risk label mapping:")
display(
    pd.crosstab(
        research_data["final_result"],
        research_data["at_risk"],
        margins=True
    )
)

In [ ]:
expected_at_risk = research_data["final_result"].map({
    "Fail": 1,
    "Withdrawn": 1,
    "Pass": 0,
    "Distinction": 0
})

print("Missing expected labels:", expected_at_risk.isna().sum())

print(
    "Incorrect at-risk labels:",
    (research_data["at_risk"] != expected_at_risk).sum()
)

print(
    "Unexpected final results:",
    (~research_data["final_result"].isin(
        ["Fail", "Withdrawn", "Pass", "Distinction"]
    )).sum()
)

In [ ]:
target_columns = {"at_risk", "final_result"}

print("Model 1 target leakage check:")
print(target_columns.intersection(X.columns))

print("\nModel 2 target leakage check:")
print(target_columns.intersection(X2.columns))

print("\nModel 3 target leakage check:")
print(target_columns.intersection(X3.columns))

In [ ]:
print("Model 1 features:")
print(X.columns.tolist())

print("\nModel 2 features:")
print(X2.columns.tolist())

print("\nModel 3 features:")
print(X3.columns.tolist())

In [ ]:
columns_to_check = [
    "clicks_first_14_days",
    "clicks_first_7_days",
    "num_of_prev_attempts",
    "studied_credits",
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

for col in columns_to_check:
    if col in model_3_data.columns:
        print(
            f"{col}: "
            f"missing = {model_3_data[col].isna().sum()}, "
            f"unique values = {model_3_data[col].nunique()}"
        )
    elif col in research_data.columns:
        print(
            f"{col}: "
            f"missing = {research_data[col].isna().sum()}, "
            f"unique values = {research_data[col].nunique()}"
        )
    else:
        print(f"{col}: NOT FOUND")

In [ ]:
categorical_columns = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(research_data[col].value_counts().sort_index())

In [ ]:
categorical_columns = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print("Categories:", sorted(research_data[col].unique()))
    print("Total records:", research_data[col].count())
    print("Missing values:", research_data[col].isna().sum())

In [ ]:
print("IMD band categories:")
print(sorted(research_data["imd_band"].unique()))

print("\nAge band categories:")
print(sorted(research_data["age_band"].unique()))

In [ ]:
print("Unknown IMD band records:")
print((research_data["imd_band"] == "?").sum())

print("\nIMD band distribution:")
print(research_data["imd_band"].value_counts().sort_index())

In [ ]:
print(model_2)

In [ ]:
print(
    "Unknown IMD records in training data:",
    (X2["imd_band"] == "?").sum()
)

print(
    "Unknown IMD records in test data:",
    (X2_test["imd_band"] == "?").sum()
)

In [ ]:
print("X2 training rows:", len(X2))
print("X2_test rows:", len(X2_test))
print("Total rows:", len(X2) + len(X2_test))

print("\nUnknown IMD values:")
print("X2:", (X2["imd_band"] == "?").sum())
print("X2_test:", (X2_test["imd_band"] == "?").sum())


In [ ]:
print("X2 shape:", X2.shape)
print("X2_test shape:", X2_test.shape)

print("\nVariables related to Model 2:")
for name in [
    "X2_train", "X_train2", "X_train_2",
    "y2_train", "y_train2", "y_train_2",
    "X2_test", "y2_test"
]:
    if name in globals():
        value = globals()[name]
        print(name, getattr(value, "shape", type(value)))
        

In [ ]:
print(
    "Unknown IMD records in training data:",
    (X2_train["imd_band"] == "?").sum()
)

print(
    "Unknown IMD records in test data:",
    (X2_test["imd_band"] == "?").sum()
)

print(
    "Total unknown IMD records across the split:",
    (X2_train["imd_band"] == "?").sum()
    + (X2_test["imd_band"] == "?").sum()
)

### Handling Unknown IMD Band Values

The dataset contained 1,111 records with an unknown IMD band, represented by the value `?`, accounting for approximately 3.4% of the 32,593 records. These records were retained rather than removed. The training set contained 894 such records, while the test set contained 217. The counts sum to the original 1,111 records. Model 2 uses one-hot encoding for categorical features, allowing the unknown IMD value to be represented as a separate category during training.

In [ ]:
print("Model 1 preprocessing:")
print(lr_pipeline.named_steps["preprocessor"])

print("\nModel 2 preprocessing:")
print(model_2.named_steps["preprocessor"])

print("\nModel 3 preprocessing:")
print(model_3.named_steps["preprocessor"])

In [ ]:

model_2.fit

In [ ]:
model_3.fit(X3_train, y3_train)

In [ ]:
model_comparison_df

### Model Comparison and Interpretation

The results show that Model 2, which includes extended student and course features, achieved the best overall performance. It obtained an accuracy of 69.2%, an F1-score of 0.708, and a ROC-AUC of 0.757. Model 1, using 14 days of learning activity and basic student/course features, achieved the highest recall at 72.0%. This means it identified a slightly larger proportion of at-risk students than Model 2.

Model 3, which used only the first seven days of learning activity, had the lowest performance across all five metrics. Its ROC-AUC was 0.715, compared with 0.737 for Model 1 and 0.757 for Model 2. This suggests that the shorter observation window may provide less predictive information, although differences in model features and experimental setup should also be considered.

Overall, the findings suggest that early learning activity and student/course characteristics can help identify students who may fail or withdraw. However, the models have limitations: their performance is moderate, and they do not identify every at-risk student. Additional validation would be needed before using these predictions to guide real student-support decisions.

In [ ]:
import matplotlib.pyplot as plt

# Set the model names as the chart labels
comparison = model_comparison_df.set_index("Model")

# Select the five evaluation metrics
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# Create a bar chart
comparison[metrics].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Performance Comparison of Student Risk Prediction Models")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Evaluation Metric")
plt.tight_layout()
plt.show()

### Model Performance Visualization

The bar chart compares the accuracy, precision, recall, F1-score, and ROC-AUC of the three prediction models. Model 2 (Extended) achieved the strongest overall performance, with the highest accuracy, precision, F1-score, and ROC-AUC. Model 1 (14-day) achieved the highest recall, meaning it identified a slightly larger proportion of at-risk students. Model 3 (7-day) had the lowest scores across all five metrics.

The results suggest that adding student and course characteristics may improve risk prediction, while using only seven days of learning activity may limit predictive performance. However, the improvements are modest, and further validation is needed before these models could be used in real educational settings.

In [ ]:
fairness_df

In [ ]:
display(fairness_df)

In [ ]:
from sklearn.metrics import recall_score, precision_score

gender_results = []

for gender, group in fairness_df.groupby("gender"):
    gender_results.append({
        "Gender": gender,
        "Students": len(group),
        "Actual at-risk students": (group["actual"] == 1).sum(),
        "Recall": recall_score(group["actual"], group["predicted"], zero_division=0),
        "Precision": precision_score(group["actual"], group["predicted"], zero_division=0)
    })

gender_fairness = pd.DataFrame(gender_results)
display(gender_fairness.round(3))

### Fairness Analysis by Gender

The model's performance differed between the two gender groups. Recall was 66.5% for female students and 74.7% for male students, representing an 8.2 percentage-point gap. This indicates that the model identified a smaller proportion of at-risk female students in the test set.

Precision was also slightly lower for female students (69.4%) than for male students (71.4%). These differences suggest that the model's prediction errors are not distributed equally across the two groups. However, these results alone do not establish the cause of the differences or prove discrimination. Further analysis would be needed to investigate possible contributing factors and evaluate whether the model could be used fairly in educational support settings.

In [ ]:
age_results = []

for age, group in fairness_df.assign(
    age_band=X2_test["age_band"].values
).groupby("age_band"):
    age_results.append({
        "Age group": age,
        "Students": len(group),
        "Actual at-risk students": (group["actual"] == 1).sum(),
        "Recall": recall_score(
            group["actual"], group["predicted"], zero_division=0
        ),
        "Precision": precision_score(
            group["actual"], group["predicted"], zero_division=0
        )
    })

age_fairness = pd.DataFrame(age_results)
display(age_fairness.round(3))

### Fairness Analysis by Age Group

The model's recall varied across age groups. Recall was 74.1% for students aged 0–35, 63.3% for students aged 35–55, and 50.0% for students aged 55 and older. This suggests that the model identified a smaller proportion of at-risk students in the older groups within this test set.

Precision was similar for the first two age groups, at 70.6% and 70.5%, respectively. The oldest group had a precision of 64.3%, but it contained only 42 students, including 18 actual at-risk students. Therefore, its results should be interpreted cautiously because a small number of predictions can substantially affect the reported metrics.

These findings highlight the importance of evaluating model performance across demographic groups rather than relying only on overall accuracy. The observed differences do not establish their cause or prove discrimination. Further testing with larger and more representative samples would be needed before drawing stronger conclusions about fairness.


## Limitations of the Study

Although the results show that early learning activity can help predict student academic outcomes, this study has several limitations.

**1. Moderate predictive performance**

The best-performing model, Model 2, achieved an ROC-AUC of 0.757 and an accuracy of approximately 69.2%. While these results show that the model can distinguish between at-risk and not-at-risk students to some extent, it still makes incorrect predictions. Therefore, its predictions should not be used as the only basis for making decisions about students.

**2. Limited early learning information**

Model 1 used activity data from the first 14 days, while Model 3 used activity data from the first 7 days. The shorter observation window may contain less information about students' learning behavior. However, the models also differ in their feature sets and experimental setup, so the performance difference cannot be attributed to the observation window alone.

**3. Differences in performance across demographic groups**

The fairness analysis showed that recall varied across gender and age groups. For example, recall was 66.5% for female students and 74.7% for male students. Recall was also lower for students aged 35–55 than for students aged 0–35. These differences suggest that the model may miss at-risk students at different rates across groups. However, the analysis does not establish the causes of these differences or prove discrimination. The oldest age group also contained only 42 students, making its results less reliable.

**4. Limited generalizability**

This study used a single educational dataset. Student characteristics, course structures, teaching methods, and learning platforms may differ across institutions. As a result, the models may not perform equally well on students from other schools or educational settings. Testing the models on additional datasets would help determine whether the findings generalize.

**5. Association does not imply causation**

The models identify patterns associated with students' final academic outcomes, but they do not explain why students fail or withdraw. For example, lower online activity may be associated with academic risk, but it does not necessarily mean that low activity causes poor outcomes. Other factors, such as personal circumstances, access to resources, and course difficulty, may also influence student performance

## Conclusion

This project investigated whether early learning activity and student/course characteristics could help identify students at risk of failing or withdrawing. Three logistic regression models were evaluated using accuracy, precision, recall, F1-score, and ROC-AUC.

Model 2, which included extended student and course features, achieved the best overall performance, with 69.2% accuracy, a 0.708 F1-score, and a 0.757 ROC-AUC. Model 1, which used the first 14 days of learning activity and basic student/course features, achieved the highest recall at 72.0%. Model 3, which used only the first seven days of activity, performed less well across all five metrics.

The fairness analysis also revealed differences in recall across gender and age groups. These findings show why overall model performance should be considered alongside the possibility that some groups of students may be missed more often than others.

Overall, the results suggest that early learning activity and student/course characteristics can provide useful signals for identifying students who may need academic support. However, the models are not accurate enough to make decisions on their own. Future work could explore additional early assessment features, test alternative algorithms, investigate performance differences across groups, and evaluate the models on other datasets. Any real-world use should support, rather than replace, educators' judgment and direct communication with students.